In [1]:
import pandas as pd
import math

# Load dataset
df = pd.read_csv("heart_disease_categorical.csv")

# Change this if your target column has a different name
target_column = "target"

# Significance level
alpha = 0.05


def manual_chi_square(data, feature, target):
    # Create observed frequency table manually using crosstab
    observed = pd.crosstab(data[feature], data[target])

    # Row totals and column totals
    row_totals = observed.sum(axis=1)
    column_totals = observed.sum(axis=0)
    grand_total = observed.values.sum()

    # Store expected frequencies
    expected = pd.DataFrame(
        0.0,
        index=observed.index,
        columns=observed.columns
    )

    # Calculate expected frequency:
    # Expected = (Row Total × Column Total) / Grand Total
    for row in observed.index:
        for col in observed.columns:
            expected.loc[row, col] = (
                row_totals[row] * column_totals[col]
            ) / grand_total

    # Calculate Chi-square step by step
    chi_square = 0

    for row in observed.index:
        for col in observed.columns:
            O = observed.loc[row, col]
            E = expected.loc[row, col]

            if E != 0:
                chi_square += ((O - E) ** 2) / E

    # Degrees of freedom
    rows = len(observed.index)
    columns = len(observed.columns)
    degrees_of_freedom = (rows - 1) * (columns - 1)

    return observed, expected, chi_square, degrees_of_freedom


# Detect categorical columns
categorical_columns = df.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Also include low-cardinality integer columns
for column in df.columns:
    if column != target_column and column not in categorical_columns:
        if df[column].nunique() <= 10:
            categorical_columns.append(column)

# Remove target from feature list
categorical_columns = [
    col for col in categorical_columns
    if col != target_column
]


# Perform Chi-square test
results = []

for feature in categorical_columns:

    observed, expected, chi_square, dfree = manual_chi_square(
        df,
        feature,
        target_column
    )

    # Critical value calculation is avoided here because
    # scipy.stats is not being used.
    #
    # For common df values, approximate critical values at α=0.05
    critical_values = {
        1: 3.841,
        2: 5.991,
        3: 7.815,
        4: 9.488,
        5: 11.070,
        6: 12.592,
        7: 14.067,
        8: 15.507,
        9: 16.919,
        10: 18.307
    }

    critical_value = critical_values.get(dfree, None)

    if critical_value is not None:
        decision = (
            "Reject H0"
            if chi_square > critical_value
            else "Fail to Reject H0"
        )
    else:
        decision = "Critical value not available"

    results.append({
        "Feature": feature,
        "Chi-Square": chi_square,
        "Degrees of Freedom": dfree,
        "Critical Value (α=0.05)": critical_value,
        "Decision": decision
    })


# Display final results
results_df = pd.DataFrame(results)

print("\n===== CHI-SQUARE TEST RESULTS =====\n")
print(results_df.to_string(index=False))


# ---------------------------------------------------------
# Show detailed calculation for every categorical feature
# ---------------------------------------------------------

for feature in categorical_columns:

    observed, expected, chi_square, dfree = manual_chi_square(
        df,
        feature,
        target_column
    )

    print("\n" + "=" * 70)
    print(f"FEATURE: {feature}")
    print("=" * 70)

    print("\nObserved Frequencies:")
    print(observed)

    print("\nExpected Frequencies:")
    print(expected.round(3))

    print("\nStep-by-step Chi-square calculation:")

    total = 0

    for row in observed.index:
        for col in observed.columns:

            O = observed.loc[row, col]
            E = expected.loc[row, col]

            if E != 0:
                contribution = ((O - E) ** 2) / E
                total += contribution

                print(
                    f"O={O}, E={E:.3f}  "
                    f"-> (O-E)^2/E = {contribution:.4f}"
                )

    print(f"\nChi-square statistic = {total:.4f}")
    print(f"Degrees of freedom = {dfree}")

    if dfree in critical_values:
        critical = critical_values[dfree]

        print(f"Critical value (α=0.05) = {critical}")

        if total > critical:
            print("Conclusion: Reject H0")
            print(
                "There is a statistically significant association "
                "between the feature and heart disease target."
            )
        else:
            print("Conclusion: Fail to Reject H0")
            print(
                "There is not enough evidence of an association "
                "between the feature and heart disease target."
            )

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_3776\111357868.py:58: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(



===== CHI-SQUARE TEST RESULTS =====

Feature  Chi-Square  Degrees of Freedom  Critical Value (α=0.05)          Decision
    sex    0.000362                   1                    3.841 Fail to Reject H0
     cp    6.659428                   3                    7.815 Fail to Reject H0
    fbs    2.560329                   1                    3.841 Fail to Reject H0
restecg    0.124149                   2                    5.991 Fail to Reject H0
  exang   19.119427                   1                    3.841         Reject H0
  slope    6.012306                   2                    5.991         Reject H0
     ca    5.638627                   3                    7.815 Fail to Reject H0
   thal    2.234412                   2                    5.991 Fail to Reject H0

FEATURE: sex

Observed Frequencies:
target   0   1
sex           
Female  11  37
Male    12  40

Expected Frequencies:
target      0      1
sex                 
Female  11.04  36.96
Male    11.96  40.04

Step-by-st